# 1. Imports

In [ ]:
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import os, re, json
import pytesseract
import datetime

def show(img, title=""):
    plt.figure(figsize=(8,4))
    if len(img.shape)==2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

- 0    Orientation and script detection (OSD) only.
- 1    Automatic page segmentation with OSD.
- 2    Automatic page segmentation, but no OSD, or OCR. (not implemented)
- 3    Fully automatic page segmentation, but no OSD. (Default)
- 4    Assume a single column of text of variable sizes.
- 5    Assume a single uniform block of vertically aligned text.
- 6    Assume a single uniform block of text.
- 7    Treat the image as a single text line.
- 8    Treat the image as a single word.
- 9    Treat the image as a single word in a circle.
- 10    Treat the image as a single character.
- 11    Sparse text. Find as much text as possible in no particular order.
- 12    Sparse text with OSD.
- 13    Raw line. Treat the image as a single text line,

# 2. Prediction

In [ ]:
DATASET_NAME = "ICDAR03/apanar"
DATASET_PATH = f"resources/datasets/{DATASET_NAME}/"
print(f"Dataset path: {DATASET_PATH}")

MARGIN = 10
CONFIDENCE_THRESHOLD = 50.0
DEBUG = False
USE_BBOXES = False

## 2.1. Prediction with Tesseract

In [ ]:
def natural_sort_key(value):
    parts = re.split(r'(\d+)', value)
    return [int(part) if part.isdigit() else part.lower() for part in parts]

def save_predictions_checkpoint(path, data):
    temp_path = f"{path}.tmp"
    with open(temp_path, "w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, ensure_ascii=False)
    os.replace(temp_path, path)

def show(img, title=""):
    plt.figure(figsize=(8,4))
    if len(img.shape)==2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
def preprocess_image(image: np.ndarray):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Binarize using Otsu's thresholding
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # --- Fix: ensure black text on white background ---
    # Assume text is the minority pixel class. If white pixels are NOT
    # the majority, the image is inverted relative to what Tesseract expects.
    white_ratio = np.count_nonzero(thresh == 255) / thresh.size
    if white_ratio < 0.5:
        thresh = cv2.bitwise_not(thresh)
    # ----------------------------------------------------

    denoised = cv2.bilateralFilter(thresh, 9, 75, 75)

    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    sharpened = cv2.filter2D(denoised, -1, kernel)

    coords = np.column_stack(np.where(thresh == 0))  # text pixels are now black (0)
    angle = cv2.minAreaRect(coords)[-1]

    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle

    (h, w) = thresh.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    deskewed = cv2.warpAffine(sharpened, M, (w, h), flags=cv2.INTER_CUBIC,
                              borderMode=cv2.BORDER_REPLICATE)

    return deskewed

In [ ]:
from codecarbon import EmissionsTracker

tracker = EmissionsTracker()
tracker.start()

In [ ]:
import json, os

BATCH_SIZE = 50
CHECKPOINT_PATH = os.path.join("resources/checkpoints/", f"{DATASET_NAME.replace('/', '_')}_pytess{'_bb' if USE_BBOXES else ''}_predictions.json")

predictions = {}
if os.path.exists(CHECKPOINT_PATH) and os.path.getsize(CHECKPOINT_PATH) > 0:
    try:
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as handle:
            predictions = json.load(handle)
        print(f"Resuming from checkpoint with {len(predictions)} processed images: {CHECKPOINT_PATH}")
    except json.JSONDecodeError:
        print(f"Checkpoint file could not be read, starting from scratch: {CHECKPOINT_PATH}")

if USE_BBOXES:
    metadata_file = json.load(open(DATASET_PATH + "data.json"))
    word_count = sum(len(coords_list) for coords_list in metadata_file.values()) 
    print(f"Metadata loaded for {word_count} words in {len(metadata_file)} images.")
    
print(f"Processing images in dataset path: {DATASET_PATH}")

image_files = sorted(
    [file for file in os.listdir(DATASET_PATH) if file.lower().endswith((".jpg", ".jpeg", ".png"))],
    key=lambda file: natural_sort_key(os.path.splitext(file)[0])
)
start_time = datetime.datetime.now()
last_checkpoint_time = start_time
for batch_start in range(0, len(image_files), BATCH_SIZE):
    batch_files = image_files[batch_start:batch_start + BATCH_SIZE]
    batch_names = [os.path.splitext(file)[0] for file in batch_files]

    if not batch_names:
        continue

    print(f"Processing batch [{batch_names[0]} to {batch_names[-1]}]")

    for file in batch_files:
        image_name = os.path.splitext(file)[0]

        if image_name in predictions:
            continue

        words = {}
        
        # Preprocessing the image
        image_path = os.path.join(DATASET_PATH, file)
        image = cv2.imread(image_path)
        if image is None:
            print(f"Could not read image: {image_path}")
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        #image = preprocess_image(image)
        
        if USE_BBOXES:
            image_meta = metadata_file.get(image_name, {})
            # Treat each ground truth key (word) and its coordinates
            for gt_key, coords_list in image_meta.items():
                for coords in coords_list:
                    word = ""
                    if len(coords) == 4:
                        x1, y1, x2, y2 = map(int, coords)
                        x1 = max(0, x1 - MARGIN)
                        y1 = max(0, y1 - MARGIN)
                        x2 = min(image.shape[1], x2 + MARGIN)
                        y2 = min(image.shape[0], y2 + MARGIN)
                        roi = image[y1:y2, x1:x2]

                    elif len(coords) == 8:
                        x1, y1, x2, y2, x3, y3, x4, y4 = map(int, coords)
                        x1 = max(0, x1 - MARGIN)
                        y1 = max(0, y1 - MARGIN)
                        x3 = min(image.shape[1], x3 + MARGIN)
                        y3 = min(image.shape[0], y3 + MARGIN)
                        roi = image[y1:y3, x1:x3]

                    elif len(coords) > 8 and len(coords) % 2 == 0:
                        x_s = [coords[i] for i in range(0, len(coords), 2)]
                        y_s = [coords[i] for i in range(1, len(coords), 2)]
                        x1, y1 = max(0, min(x_s) - MARGIN), max(0, min(y_s) - MARGIN)
                        x2, y2 = min(image.shape[1], max(x_s) + MARGIN), min(image.shape[0], max(y_s) + MARGIN)
                        roi = image[y1:y2, x1:x2]

                    else:
                        print(f"Unexpected number of coordinates for {image_name} - {gt_key}: {len(coords)}")
                        continue

                    if roi is None or roi.size == 0:
                        #print(f"ROI extraction failed for {image_name} - {gt_key}: {coords}")
                        continue

                    if DEBUG:
                        debug_dir = os.path.join(DATASET_PATH, "debug")
                        os.makedirs(debug_dir, exist_ok=True)
                        roi_path = os.path.join(debug_dir, f"{image_name}_{gt_key}_roi.jpg")
                        if not cv2.imwrite(roi_path, cv2.cvtColor(roi, cv2.COLOR_RGB2BGR)):
                            # Could not save because the path contains invalid characters.
                            for char in r'<>:"/\|?*':
                                gt_key = gt_key.replace(char, '')
                            roi_path = os.path.join(debug_dir, f"{image_name}_{gt_key}_roi.jpg")
                            cv2.imwrite(roi_path, cv2.cvtColor(roi, cv2.COLOR_RGB2BGR))

                    config = ' --psm 8' if len(gt_key.split()) == 1 else ' --psm 7'
                    exec_start_time = datetime.datetime.now()
                    data = pytesseract.image_to_data(
                        roi,
                        output_type=pytesseract.Output.DATAFRAME,
                        config=config
                    )
                    exec_end_time = datetime.datetime.now()
                    process_time = exec_end_time - exec_start_time
                    process_time = round(process_time.total_seconds(), 4)
                    cdd_words = {}

                    for i in range(len(data['text'])):
                        conf = data.iloc[i]["conf"]
                        if conf == -1 or not str(data.iloc[i]["text"]).strip():
                            continue
                        else:
                            word = str(data.iloc[i]["text"]).strip()
                            # Remove unicode characters that are not in the allowed set
                            word = re.sub(r'[^\w\s.,!?;:(){}\[\]\'"-]', '', word)
                            cdd_words[conf] = word
    
                    if cdd_words:
                        # Get the word with the highest confidence and its confidence score
                        conf = max(cdd_words.keys())
                        word = cdd_words[conf]
                    else:
                        word = None
                        conf = 0.0
                    word_details = (word, conf, process_time) if word else ("", 0.0, process_time)

                    if gt_key in words:
                        if isinstance(words[gt_key], list):
                            words[gt_key].append(word_details)
                        else:
                            words[gt_key] = [words[gt_key], word_details]
                    else:
                        words[gt_key] = word_details
        else:
            words = []
            exec_start_time = datetime.datetime.now()
            data = pytesseract.image_to_data(
                image,
                output_type=pytesseract.Output.DATAFRAME,
            )
            exec_end_time = datetime.datetime.now()
            process_time = exec_end_time - exec_start_time
            process_time = round(process_time.total_seconds(), 4)
            cdd_words = {}

            for i in range(len(data['text'])):
                conf = round(float(data.iloc[i]["conf"]), 4)
                if conf == -1 or not str(data.iloc[i]["text"]).strip():
                    continue
                else:
                    word = str(data.iloc[i]["text"]).strip()
                    # Remove unicode characters that are not in the allowed set
                    word = re.sub(r'[^\w\s.,!?;:(){}\[\]\'"-]', '', word)
                    cdd_words[word] = conf
                    
            for word, conf in cdd_words.items():
                conf = float(conf)
                if conf >= CONFIDENCE_THRESHOLD:
                    words.append((word, conf, process_time))
                
        predictions[image_name] = words

    save_predictions_checkpoint(CHECKPOINT_PATH, predictions)
    step_time = datetime.datetime.now()
    elapsed_time = step_time - last_checkpoint_time
    minutes, seconds = divmod(elapsed_time.total_seconds(), 60)
    last_checkpoint_time = step_time

    print(f"Checkpoint saved after batch [{batch_names[0]} to {batch_names[-1]}]: {len(predictions)} images in {int(minutes)} minutes and {int(seconds)} seconds.")

end_time = datetime.datetime.now()
print(f"Predictions completed for {len(predictions)} images.")
total_time = end_time - start_time
minutes, seconds = divmod(total_time.total_seconds(), 60)
print(f"Total time taken: {int(minutes)} minutes and {int(seconds)} seconds.")

## 2.2. Save predictions to JSON

In [ ]:
print(f"Saving predictions to output/predicitions/{DATASET_NAME}_pytess{"_bb" if USE_BBOXES else ""}.json")
with open(f"output/predictions/{DATASET_NAME}_pytess{"_bb" if USE_BBOXES else ""}.json", "w") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

In [ ]:
emissions = tracker.stop()
print(f"Emissions: {emissions} kg CO₂")